# 5. Monte Carlo Methods

Bu notebook, Sutton & Barto kitabının 5. bölümünü kapsar.

## İçindekiler
1. MC'ye Giriş
2. MC Prediction
3. MC Control
4. On-policy vs Off-policy
5. Importance Sampling

## 5.1 Monte Carlo Nedir?

**Monte Carlo (MC)** metodları, **deneyimden** (experience) öğrenir - model gerektirmez!

### DP vs MC

| Özellik | DP | MC |
|---------|----|----|  
| Model | Gerekli | Gerekli değil |
| Update | Her state için | Episode sonunda |
| Bootstrap | Evet | Hayır |

### MC'nin Temel Fikri

Value'yu **sample returns**'ların ortalaması olarak tahmin et:

$$V(s) \approx \frac{1}{N(s)} \sum_{i=1}^{N(s)} G_i(s)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Tuple, Dict

class BlackjackEnv:
    """
    Simplified Blackjack environment.
    State: (player_sum, dealer_showing, usable_ace)
    Actions: 0 = stick, 1 = hit
    """
    
    def __init__(self):
        self.action_space = [0, 1]  # stick, hit
    
    def draw_card(self):
        """Draw a card (1-10, face cards = 10)."""
        card = min(np.random.randint(1, 14), 10)
        return card
    
    def draw_hand(self):
        """Draw initial hand."""
        return [self.draw_card(), self.draw_card()]
    
    def usable_ace(self, hand):
        """Check if hand has usable ace."""
        return 1 in hand and sum(hand) + 10 <= 21
    
    def sum_hand(self, hand):
        """Sum of hand, ace = 11 if usable."""
        if self.usable_ace(hand):
            return sum(hand) + 10
        return sum(hand)
    
    def is_bust(self, hand):
        return self.sum_hand(hand) > 21
    
    def reset(self):
        """Start new episode."""
        self.player = self.draw_hand()
        self.dealer = self.draw_hand()
        
        # Ensure starting sum is 12-21
        while self.sum_hand(self.player) < 12:
            self.player.append(self.draw_card())
        
        return self._get_state()
    
    def _get_state(self):
        return (
            self.sum_hand(self.player),
            self.dealer[0],  # Dealer showing
            self.usable_ace(self.player)
        )
    
    def step(self, action):
        """Take action, return (state, reward, done)."""
        if action == 1:  # Hit
            self.player.append(self.draw_card())
            
            if self.is_bust(self.player):
                return self._get_state(), -1, True
            else:
                return self._get_state(), 0, False
        
        else:  # Stick - dealer's turn
            while self.sum_hand(self.dealer) < 17:
                self.dealer.append(self.draw_card())
            
            player_sum = self.sum_hand(self.player)
            dealer_sum = self.sum_hand(self.dealer)
            
            if self.is_bust(self.dealer) or player_sum > dealer_sum:
                reward = 1
            elif player_sum < dealer_sum:
                reward = -1
            else:
                reward = 0
            
            return self._get_state(), reward, True

# Test
env = BlackjackEnv()
state = env.reset()
print(f"Initial state: {state}")
print(f"(player_sum={state[0]}, dealer_showing={state[1]}, usable_ace={state[2]})")

## 5.2 MC Prediction (Policy Evaluation)

Verilen bir policy $\pi$ için $V^\pi$ veya $Q^\pi$ tahmin et.

### First-Visit MC

Her episode'da state $s$'in **ilk** ziyaretindeki return'ü kullan:

```
For each episode:
    Generate episode following π
    For each state s in episode (first visit only):
        G ← return following first visit to s
        Append G to Returns(s)
        V(s) ← average(Returns(s))
```

### Every-Visit MC

Her ziyareti say (birden fazla ziyaret varsa).

In [ ]:
def generate_episode(env, policy):
    """
    Policy'yi takip ederek bir episode oluştur.
    
    Returns:
        episode: List of (state, action, reward)
    """
    episode = []
    state = env.reset()
    
    while True:
        action = policy(state)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        
        if done:
            break
        state = next_state
    
    return episode

# Simple policy: stick if sum >= 20
def simple_policy(state):
    player_sum, dealer_showing, usable_ace = state
    return 0 if player_sum >= 20 else 1

# Test
episode = generate_episode(env, simple_policy)
print("Episode:")
for s, a, r in episode:
    print(f"  State: {s}, Action: {'stick' if a==0 else 'hit'}, Reward: {r}")

In [ ]:
def mc_prediction_v(env, policy, n_episodes=10000, gamma=1.0):
    """
    First-Visit MC Prediction for V(s).
    """
    V = defaultdict(float)
    returns = defaultdict(list)
    
    for _ in range(n_episodes):
        episode = generate_episode(env, policy)
        
        # Calculate returns
        G = 0
        visited_states = set()
        
        # Traverse episode backwards
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            # First-visit check
            if state not in visited_states:
                visited_states.add(state)
                returns[state].append(G)
                V[state] = np.mean(returns[state])
    
    return V, returns

V, returns = mc_prediction_v(env, simple_policy, n_episodes=50000)
print(f"Estimated V for {len(V)} states")

In [ ]:
def plot_value_function(V, title="Value Function"):
    """Blackjack value function'ı görselleştir."""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for idx, usable_ace in enumerate([False, True]):
        ax = axes[idx]
        
        # Create grid
        player_range = range(12, 22)
        dealer_range = range(1, 11)
        
        grid = np.zeros((len(player_range), len(dealer_range)))
        
        for i, player in enumerate(player_range):
            for j, dealer in enumerate(dealer_range):
                state = (player, dealer, usable_ace)
                grid[i, j] = V.get(state, 0)
        
        im = ax.imshow(grid, cmap='RdYlGn', origin='lower', 
                       extent=[0.5, 10.5, 11.5, 21.5], aspect='auto',
                       vmin=-1, vmax=1)
        
        ax.set_xlabel('Dealer Showing')
        ax.set_ylabel('Player Sum')
        ax.set_title(f'Usable Ace: {usable_ace}')
        plt.colorbar(im, ax=ax)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_value_function(V, "MC Prediction: V(s) for Simple Policy")

## 5.3 MC Control

Optimal policy bulmak için MC kullan.

### MC with Exploring Starts

Her state-action çiftiyle başlama şansı olsun.

### MC Control without Exploring Starts

**ε-greedy** policy kullan:

$$\pi(a|s) = \begin{cases} 1 - \epsilon + \frac{\epsilon}{|A|} & \text{if } a = \arg\max_a Q(s,a) \\ \frac{\epsilon}{|A|} & \text{otherwise} \end{cases}$$

In [ ]:
def mc_control_epsilon_greedy(env, n_episodes=100000, gamma=1.0, epsilon=0.1):
    """
    On-policy MC Control with ε-greedy policy.
    """
    Q = defaultdict(lambda: np.zeros(2))  # 2 actions
    returns = defaultdict(list)
    
    def epsilon_greedy_policy(state):
        if np.random.random() < epsilon:
            return np.random.randint(2)
        else:
            return np.argmax(Q[state])
    
    for ep in range(n_episodes):
        # Generate episode
        episode = generate_episode(env, epsilon_greedy_policy)
        
        # Calculate returns and update Q
        G = 0
        visited_sa = set()
        
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            sa = (state, action)
            if sa not in visited_sa:
                visited_sa.add(sa)
                returns[sa].append(G)
                Q[state][action] = np.mean(returns[sa])
    
    # Extract greedy policy
    policy = {}
    for state in Q:
        policy[state] = np.argmax(Q[state])
    
    return Q, policy

Q, optimal_policy = mc_control_epsilon_greedy(env, n_episodes=100000)
print(f"Learned Q for {len(Q)} states")

In [ ]:
def plot_policy(policy, title="Policy"):
    """Blackjack policy'sini görselleştir."""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for idx, usable_ace in enumerate([False, True]):
        ax = axes[idx]
        
        player_range = range(12, 22)
        dealer_range = range(1, 11)
        
        grid = np.zeros((len(player_range), len(dealer_range)))
        
        for i, player in enumerate(player_range):
            for j, dealer in enumerate(dealer_range):
                state = (player, dealer, usable_ace)
                grid[i, j] = policy.get(state, 1)  # Default: hit
        
        im = ax.imshow(grid, cmap='RdYlGn', origin='lower',
                       extent=[0.5, 10.5, 11.5, 21.5], aspect='auto',
                       vmin=0, vmax=1)
        
        ax.set_xlabel('Dealer Showing')
        ax.set_ylabel('Player Sum')
        ax.set_title(f'Usable Ace: {usable_ace}')
        
        # Legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor='red', label='Stick (0)'),
                          Patch(facecolor='green', label='Hit (1)')]
        ax.legend(handles=legend_elements, loc='upper right')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_policy(optimal_policy, "MC Control: Optimal Policy")

## 5.4 On-policy vs Off-policy

### On-policy
- **Aynı** policy'yi hem öğrenmek hem de veri toplamak için kullan
- Öğrenilen policy = behavior policy
- Örnek: ε-greedy MC Control

### Off-policy
- **Farklı** policy'ler: target policy (öğrenilen) vs behavior policy (veri toplayan)
- Daha esnek, ama daha karmaşık
- **Importance Sampling** gerektirir

## 5.5 Importance Sampling

Bir dağılımdan sample alıp başka bir dağılımın expected value'sunu tahmin etme tekniği.

### Importance Sampling Ratio

$$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

Burada:
- $\pi$: Target policy
- $b$: Behavior policy

### Ordinary Importance Sampling

$$V(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{|\mathcal{T}(s)|}$$

### Weighted Importance Sampling

$$V(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1}}$$

Weighted IS genelde daha düşük variance'a sahiptir.

In [ ]:
def mc_off_policy_prediction(env, target_policy, n_episodes=10000, gamma=1.0):
    """
    Off-policy MC Prediction using Weighted Importance Sampling.
    Behavior policy: random
    """
    Q = defaultdict(lambda: np.zeros(2))
    C = defaultdict(lambda: np.zeros(2))  # Cumulative weights
    
    def behavior_policy(state):
        """Random behavior policy."""
        return np.random.randint(2)
    
    for _ in range(n_episodes):
        # Generate episode with behavior policy
        episode = generate_episode(env, behavior_policy)
        
        G = 0
        W = 1.0  # Importance sampling weight
        
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            C[state][action] += W
            Q[state][action] += (W / C[state][action]) * (G - Q[state][action])
            
            # Target policy action
            target_action = target_policy(state)
            
            if action != target_action:
                break  # W would be 0, stop updating
            
            # Update W: π(a|s) / b(a|s)
            # π is deterministic (1 for target action), b is random (0.5)
            W *= 1.0 / 0.5
    
    return Q

# Target: stick on 20 or 21
def target_policy(state):
    return 0 if state[0] >= 20 else 1

Q_off = mc_off_policy_prediction(env, target_policy, n_episodes=50000)
print(f"Off-policy Q estimated for {len(Q_off)} states")

In [ ]:
# Variance comparison: Ordinary vs Weighted IS
def compare_importance_sampling(env, n_runs=100, episode_counts=[1, 10, 100, 1000, 10000]):
    """
    Compare ordinary and weighted IS estimates.
    """
    # Single state to estimate
    target_state = (13, 2, True)  # Example state
    
    def target_policy(state):
        return 0 if state[0] >= 20 else 1
    
    def behavior_policy(state):
        return np.random.randint(2)
    
    results = {'ordinary': [], 'weighted': []}
    
    for n_episodes in episode_counts:
        ordinary_estimates = []
        weighted_estimates = []
        
        for run in range(n_runs):
            # Collect episodes starting from target_state
            ordinary_returns = []
            weighted_num = 0
            weighted_denom = 0
            
            for _ in range(n_episodes):
                # Force start from target state (simplified)
                env.reset()
                env.player = [3, 10] if target_state[2] else [3, 10]  # Simplified
                
                episode = generate_episode(env, behavior_policy)
                
                if len(episode) > 0:
                    G = sum([r for _, _, r in episode])
                    
                    # Calculate importance ratio
                    rho = 1.0
                    for state, action, _ in episode:
                        target_action = target_policy(state)
                        if action == target_action:
                            rho *= 2.0  # π/b = 1/0.5
                        else:
                            rho = 0
                            break
                    
                    ordinary_returns.append(rho * G)
                    weighted_num += rho * G
                    weighted_denom += rho
            
            if ordinary_returns:
                ordinary_estimates.append(np.mean(ordinary_returns))
            if weighted_denom > 0:
                weighted_estimates.append(weighted_num / weighted_denom)
        
        if ordinary_estimates:
            results['ordinary'].append(np.var(ordinary_estimates))
        if weighted_estimates:
            results['weighted'].append(np.var(weighted_estimates))
    
    return results, episode_counts

print("Importance Sampling comparison (simplified demo)")

## 5.6 Incremental Implementation

Her episode'dan sonra tüm return'leri saklamak yerine **incremental** güncelleme:

$$V(S_t) \leftarrow V(S_t) + \alpha [G_t - V(S_t)]$$

Bu, sample average'ın bir genellemesidir:
- $\alpha = 1/n$: Sample average (non-stationary değil)
- $\alpha$ sabit: Non-stationary problemler için

In [ ]:
def mc_control_incremental(env, n_episodes=100000, gamma=1.0, alpha=0.01, epsilon=0.1):
    """
    MC Control with constant-α (incremental updates).
    """
    Q = defaultdict(lambda: np.zeros(2))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(2)
        return np.argmax(Q[state])
    
    rewards_per_episode = []
    
    for ep in range(n_episodes):
        episode = generate_episode(env, epsilon_greedy)
        
        episode_reward = sum([r for _, _, r in episode])
        rewards_per_episode.append(episode_reward)
        
        G = 0
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            # Incremental update
            Q[state][action] += alpha * (G - Q[state][action])
    
    return Q, rewards_per_episode

Q_inc, rewards = mc_control_incremental(env, n_episodes=100000)

# Learning curve
window = 1000
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 5))
plt.plot(smoothed)
plt.xlabel('Episode')
plt.ylabel('Average Reward (per 1000 episodes)')
plt.title('MC Control Learning Curve')
plt.grid(True, alpha=0.3)
plt.show()

## Özet

| Kavram | Açıklama |
|--------|----------|
| **MC** | Episode sonunda öğrenme, model-free |
| **First-Visit MC** | İlk ziyareti say |
| **Every-Visit MC** | Tüm ziyaretleri say |
| **MC Control** | Policy optimization (ε-greedy) |
| **On-policy** | Behavior = Target |
| **Off-policy** | Behavior ≠ Target, IS gerekli |

### MC'nin Avantajları
- Model gerektirmez
- Basit ve anlaşılır
- Episode-based tasks için uygun

### MC'nin Dezavantajları
- Episode **bitmeli** (continuing tasks için uygun değil)
- Yüksek variance (tüm episode boyunca)

### Sonraki Notebook
**06 - Temporal Difference Learning**: TD(0), SARSA, Q-Learning